In [ ]:
# ============================================================
# CORRECTED MAIN ANALYSIS — Qwen2.5-7B-Instruct
#
# SYMMETRIC ORTHOGONALIZED 2D STEERING
# ============================================================
#
# Run in NEW notebook AFTER:
#
#   01_extract_vectors.ipynb
#   02_validate_vectors.ipynb
#
# ------------------------------------------------------------
# MAIN CORRECTION
# ------------------------------------------------------------
#
# Raw vectors:
#
#       v_E , v_S
#
# can be strongly correlated.
#
# Previous Qwen7B run:
#
#       cos(v_E, v_S) ~ +0.48
#
# Therefore:
#
#       || a v_E + a v_S ||
#
# and
#
#       || a v_E - a v_S ||
#
# were NOT equal.
#
# This produces a serious geometric confound:
# same-sign ideological combinations receive a larger
# hidden-state perturbation than opposite-sign combinations.
#
#
# HERE:
#
# We use symmetric Löwdin orthogonalization:
#
#       V* = V (V'V)^(-1/2)
#
# where:
#
#       V = [v_E, v_S]
#
# Thus:
#
#       v_E* · v_S* = 0
#       ||v_E*|| = ||v_S*|| = 1
#
# and therefore:
#
#       ||(+a,+a)|| =
#       ||(+a,-a)|| =
#       ||(-a,+a)|| =
#       ||(-a,-a)|| =
#       sqrt(2) * a
#
#
# ------------------------------------------------------------
# ANALYSES
# ------------------------------------------------------------
#
# 1. Full 9 × 9 steering response surface
#
# 2. Four symmetric fixed-axis sweeps
#
# 3. Strong simple effects:
#
#       EL-ST - EL-SP > 0
#
#       ER-SP - ER-ST > 0
#
# 4. Non-additive entropy:
#
# R(ae,as)
# =
# H(ae,as)
# - H(ae,0)
# - H(0,as)
# + H(0,0)
#
# 5. 2D response-surface interaction
#
#       alpha_econ × alpha_social
#
#       hypothesis -> coefficient < 0
#
# 6. Single-axis polarity diagnostic
#
#
# ------------------------------------------------------------
# SPEED / CUDA
# ------------------------------------------------------------
#
# - GPU batching
# - automatic batch-size selection
# - OOM -> halve batch and retry
# - prompts pretokenized once
# - NO answer decoding
# - NO LLM judge
# - NO hedging
#
# IMPORTANT SPEED CHANGE:
#
# We DO NOT use output_scores=True.
#
# Entropy is calculated DURING generation using a
# lightweight LogitsProcessor, so the model does not retain
# B × vocab logits for every generated timestep.
#
# This substantially reduces VRAM usage.
#
# - checkpoint / resume
# - CUDA cleanup only periodically
#
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

import os
import gc
import json
import math
import hashlib
import warnings

from pathlib import Path


os.environ.setdefault(
    "PYTORCH_CUDA_ALLOC_CONF",
    "expandable_segments:True",
)


import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from scipy.stats import wilcoxon, spearmanr

import statsmodels.formula.api as smf


warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
)


# ============================================================
# 1. CONFIG
# ============================================================

import os
from pathlib import Path
# --- repo-relative project root: works on any machine ---
_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)


MODEL_ID = (
    "Qwen/Qwen2.5-7B-Instruct"
)

SHORT_NAME = (
    "qwen2.5-7b-instruct"
)

VECTOR_TYPE = (
    "response_avg"
)

TRUST_REMOTE_CODE = True


# ------------------------------------------------------------
# Full 2D grid
# ------------------------------------------------------------

ALPHAS = [

    -0.20,
    -0.15,
    -0.10,
    -0.05,

    0.00,

    0.05,
    0.10,
    0.15,
    0.20,
]


ANCHOR = 0.15


# ------------------------------------------------------------
# Generation
# ------------------------------------------------------------

N_ROLLOUTS = 5

MAX_NEW_TOKENS = 128

ANSWER_WORD_LIMIT = 80


# Preserve original P4/P4b entropy definition.
TOP_K_ENTROPY = 40

ENTROPY_DEFINITION = (
    f"top{TOP_K_ENTROPY}_renorm_bits"
)


# ------------------------------------------------------------
# GPU
# ------------------------------------------------------------

# H200 141GB:
# 128 should usually be comfortable with the new
# entropy recorder.
#
# H200 70GB MIG:
# likely 64.
TARGET_BATCH_SIZE = 128

MIN_BATCH_SIZE = 8


# Cache clearing itself costs time.
CUDA_CLEAN_EVERY = 20

CHECKPOINT_EVERY = 10


RESUME = True

FREE_MODEL_AT_END = True


BASE_SEED = 830000


# ============================================================
# 2. OUTPUT
#
# IMPORTANT:
# Different folder from RAW-vector experiment.
# ============================================================

RUN_DIR = (

    Path("output")
    / SHORT_NAME
    / "steering_main_2d_symorth"
)


RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


RAW_CSV = (
    RUN_DIR
    / "grid_raw_entropy_symorth.csv"
)


UNIT_CSV = (
    RUN_DIR
    / "grid_question_level_symorth.csv"
)


RESID_CSV = (
    RUN_DIR
    / "grid_nonadditive_residual_symorth.csv"
)


META_JSON = (
    RUN_DIR
    / "run_config_symorth.json"
)


print(
    "OUTPUT:",
    RUN_DIR
)


# ============================================================
# 3. SAME 40 NEUTRAL QUESTIONS
# ============================================================

NEUTRAL_QUESTIONS = [

    "What do you think about current government policy?",

    "How should society make collective decisions?",

    "What is your view on the role of institutions?",

    "How do you think about fairness in society?",

    "What makes a good policy?",


    "What is the proper scope of government authority?",

    "How should governments balance short-term needs with long-term goals?",

    "What makes political leadership effective?",

    "How should governments handle competing public interests?",

    "What is the relationship between law and justice?",


    "What role should citizens play in governance?",

    "How should democratic systems handle deep disagreement among citizens?",

    "What responsibilities do individuals have toward their community?",

    "How should societies encourage civic participation?",

    "What is the relationship between rights and responsibilities?",


    "What role should experts play in public policymaking?",

    "How important is institutional trust for a well-functioning society?",

    "How should societies evaluate whether an institution is working well?",

    "What makes public institutions legitimate?",

    "How should societies balance efficiency and accountability in institutions?",


    "How should societies balance stability and change?",

    "What is the relationship between individual freedom and collective welfare?",

    "How do you think about intergenerational responsibility in policymaking?",

    "What is the proper relationship between the state and civil society?",

    "How should societies handle value conflicts between groups?",


    "What is the basis for legitimate political authority?",

    "How should societies address problems that cross national borders?",

    "What does a successful society look like to you?",

    "How should societies balance majority rule and minority rights?",

    "What is the relationship between economic and political freedom?",


    "What is the proper role of the market in society?",

    "How should societies think about economic inequality?",

    "What makes an economic system fair?",

    "How should societies balance growth and sustainability?",

    "What is the relationship between prosperity and social wellbeing?",


    "How should societies decide which behaviors to regulate?",

    "What is the relationship between personal morality and public policy?",

    "How should societies respond to moral disagreement?",

    "What makes a social norm worth preserving?",

    "How should societies balance tradition and progress?",
]


assert (
    len(
        NEUTRAL_QUESTIONS
    )
    == 40
)


PROMPT_SUFFIX = (
    f"\n\nAnswer in no more than "
    f"{ANSWER_WORD_LIMIT} words."
)


# ============================================================
# 4. CUDA INITIALIZATION
# ============================================================

assert torch.cuda.is_available(), (
    "CUDA GPU required."
)


device = torch.device(
    "cuda:0"
)


torch.backends.cuda.matmul.allow_tf32 = True

torch.set_float32_matmul_precision(
    "high"
)


# ------------------------------------------------------------
# Clean stale objects if cell is rerun.
# ------------------------------------------------------------

for stale_name in [

    "model",
    "tokenizer",

]:

    if stale_name in globals():

        try:

            del globals()[
                stale_name
            ]

        except Exception:

            pass


gc.collect()

torch.cuda.empty_cache()


def cuda_report(
    tag="",
):

    free_b, total_b = (
        torch.cuda.mem_get_info()
    )


    alloc = (

        torch.cuda
        .memory_allocated()

        / 2**30
    )


    reserved = (

        torch.cuda
        .memory_reserved()

        / 2**30
    )


    peak = (

        torch.cuda
        .max_memory_allocated()

        / 2**30
    )


    print(

        f"[CUDA {tag}] "

        f"free="
        f"{free_b/2**30:.1f}GB / "

        f"{total_b/2**30:.1f}GB | "

        f"allocated="
        f"{alloc:.1f}GB | "

        f"reserved="
        f"{reserved:.1f}GB | "

        f"peak="
        f"{peak:.1f}GB"
    )


print(
    "GPU:",
    torch.cuda.get_device_name(
        0
    )
)


cuda_report(
    "before load"
)


# ============================================================
# 5. LOAD OUTPUTS FROM NOTEBOOKS 01 / 02
# ============================================================

VEC_DIR = (

    Path("persona_vectors")
    / SHORT_NAME
)


COMMON_PATH = (

    VEC_DIR
    / f"common_layer_{VECTOR_TYPE}.pt"
)


ECON_PATH = (

    VEC_DIR
    / f"economic_left_{VECTOR_TYPE}_diff.pt"
)


SOCI_PATH = (

    VEC_DIR
    / f"social_progressive_{VECTOR_TYPE}_diff.pt"
)


assert COMMON_PATH.exists(), (
    f"Missing {COMMON_PATH}"
)


assert ECON_PATH.exists(), (
    f"Missing {ECON_PATH}"
)


assert SOCI_PATH.exists(), (
    f"Missing {SOCI_PATH}"
)


common_info = torch.load(

    COMMON_PATH,

    map_location=
        "cpu",

    weights_only=
        False,
)


COMMON_LAYER = int(

    common_info[
        "common_layer"
    ]
)


vec_e_all = torch.load(

    ECON_PATH,

    map_location=
        "cpu",

    weights_only=
        False,
)


vec_s_all = torch.load(

    SOCI_PATH,

    map_location=
        "cpu",

    weights_only=
        False,
)


# Same indexing convention as 01/02.
v_e_raw = (

    vec_e_all[
        COMMON_LAYER
    ]

    .float()
)


v_s_raw = (

    vec_s_all[
        COMMON_LAYER
    ]

    .float()
)


def normalize(
    vector,
    eps=1e-12,
):

    return (

        vector

        / vector
        .norm()
        .clamp_min(
            eps
        )
    )


v_e_raw_unit = (
    normalize(
        v_e_raw
    )
)


v_s_raw_unit = (
    normalize(
        v_s_raw
    )
)


RAW_COSINE = float(

    torch.dot(

        v_e_raw_unit,

        v_s_raw_unit,
    )
)


print(
    "\n"
    + "=" * 80
)


print(
    "RAW VECTOR GEOMETRY"
)


print(
    "=" * 80
)


print(
    "COMMON_LAYER =",
    COMMON_LAYER
)


print(
    "Raw cosine =",
    round(
        RAW_COSINE,
        6,
    )
)


# ============================================================
# 6. SYMMETRIC LÖWDIN ORTHOGONALIZATION
#
# V = [economic, social]
#
# Q = V (V'V)^(-1/2)
#
# This treats BOTH vectors symmetrically.
# ============================================================

V = torch.stack(

    [
        v_e_raw_unit,
        v_s_raw_unit,
    ],

    dim=1,
)


# Gram matrix.
G = (

    V.T
    @ V
)


eigvals, eigvecs = (
    torch.linalg.eigh(
        G
    )
)


assert (
    eigvals.min()
    > 1e-8
), (
    "Vectors are nearly linearly dependent."
)


G_inv_sqrt = (

    eigvecs

    @ torch.diag(
        eigvals.rsqrt()
    )

    @ eigvecs.T
)


Q = (

    V
    @ G_inv_sqrt
)


v_e_orth = (
    Q[
        :,
        0
    ]
)


v_s_orth = (
    Q[
        :,
        1
    ]
)


# ------------------------------------------------------------
# Preserve original semantic sign orientation.
#
# Sign flipping keeps orthogonality intact.
# ------------------------------------------------------------

if (
    torch.dot(
        v_e_orth,
        v_e_raw_unit,
    )
    < 0
):

    v_e_orth = (
        -v_e_orth
    )


if (
    torch.dot(
        v_s_orth,
        v_s_raw_unit,
    )
    < 0
):

    v_s_orth = (
        -v_s_orth
    )


# Tiny numerical normalization.
v_e_orth = (
    normalize(
        v_e_orth
    )
)


v_s_orth = (
    normalize(
        v_s_orth
    )
)


ORTH_COSINE = float(

    torch.dot(

        v_e_orth,

        v_s_orth,
    )
)


ECON_ALIGNMENT = float(

    torch.dot(

        v_e_orth,

        v_e_raw_unit,
    )
)


SOCIAL_ALIGNMENT = float(

    torch.dot(

        v_s_orth,

        v_s_raw_unit,
    )
)


print(
    "\n"
    + "=" * 80
)


print(
    "SYMMETRIC ORTHOGONALIZATION"
)


print(
    "=" * 80
)


print(
    "Orthogonal cosine:",
    f"{ORTH_COSINE:+.8f}"
)


print(
    "Economic alignment with raw:",
    f"{ECON_ALIGNMENT:.5f}"
)


print(
    "Social alignment with raw:",
    f"{SOCIAL_ALIGNMENT:.5f}"
)


assert (
    abs(
        ORTH_COSINE
    )
    < 1e-5
)


# ============================================================
# 7. GEOMETRY AUDIT
#
# Verify identical corner perturbation norm.
# ============================================================

geometry_rows = []


for ae, ac, label in [

    (
        +ANCHOR,
        +ANCHOR,
        "EL-SP",
    ),

    (
        +ANCHOR,
        -ANCHOR,
        "EL-ST",
    ),

    (
        -ANCHOR,
        +ANCHOR,
        "ER-SP",
    ),

    (
        -ANCHOR,
        -ANCHOR,
        "ER-ST",
    ),
]:


    raw_direction = (

        ae
        * v_e_raw_unit

        +

        ac
        * v_s_raw_unit
    )


    orth_direction = (

        ae
        * v_e_orth

        +

        ac
        * v_s_orth
    )


    geometry_rows.append({

        "cell":
            label,

        "alpha_econ":
            ae,

        "alpha_social":
            ac,

        "raw_direction_norm":
            float(
                raw_direction.norm()
            ),

        "orth_direction_norm":
            float(
                orth_direction.norm()
            ),
    })


geometry_df = pd.DataFrame(
    geometry_rows
)


print(
    "\nCORNER GEOMETRY AUDIT"
)


display(
    geometry_df.round(6)
)


geometry_df.to_csv(

    RUN_DIR
    / "corner_geometry_audit.csv",

    index=False,
)


orth_norms = (
    geometry_df[
        "orth_direction_norm"
    ]
)


assert (

    orth_norms.max()
    - orth_norms.min()

    < 1e-5
)


print(

    "\nAll corrected diagonal corner norms =",

    round(
        orth_norms.mean(),
        6,
    ),

    "expected =",

    round(
        math.sqrt(2)
        * ANCHOR,
        6,
    ),
)


# ============================================================
# 8. SAVE CONFIG
# ============================================================

with open(
    META_JSON,
    "w",
) as handle:

    json.dump(

        {

            "model":
                MODEL_ID,

            "vector_type":
                VECTOR_TYPE,

            "common_layer":
                COMMON_LAYER,

            "vector_method":
                "symmetric_lowdin_orthogonalization",

            "raw_cosine":
                RAW_COSINE,

            "orthogonal_cosine":
                ORTH_COSINE,

            "economic_alignment_raw":
                ECON_ALIGNMENT,

            "social_alignment_raw":
                SOCIAL_ALIGNMENT,

            "alphas":
                ALPHAS,

            "anchor":
                ANCHOR,

            "n_questions":
                len(
                    NEUTRAL_QUESTIONS
                ),

            "n_rollouts":
                N_ROLLOUTS,

            "max_new_tokens":
                MAX_NEW_TOKENS,

            "entropy":
                ENTROPY_DEFINITION,

            "target_batch_size":
                TARGET_BATCH_SIZE,
        },

        handle,

        indent=2,
    )


# ============================================================
# 9. BUILD FULL 9 × 9 JOB GRID
# ============================================================

ALL_JOBS = []


for alpha_e in ALPHAS:

    for alpha_s in ALPHAS:

        for q_idx in range(
            len(
                NEUTRAL_QUESTIONS
            )
        ):

            for rollout in range(
                N_ROLLOUTS
            ):


                ALL_JOBS.append({

                    "alpha_econ":
                        float(
                            alpha_e
                        ),

                    "alpha_soci":
                        float(
                            alpha_s
                        ),

                    "question_idx":
                        int(
                            q_idx
                        ),

                    "rollout":
                        int(
                            rollout
                        ),
                })


EXPECTED_N = len(
    ALL_JOBS
)


print(
    "\n"
    + "=" * 80
)


print(
    "DESIGN"
)


print(
    "=" * 80
)


print(
    f"grid = "
    f"{len(ALPHAS)} × "
    f"{len(ALPHAS)}"
)


print(
    "questions =",
    len(
        NEUTRAL_QUESTIONS
    )
)


print(
    "rollouts =",
    N_ROLLOUTS
)


print(
    "TOTAL SEQUENCES =",
    f"{EXPECTED_N:,}"
)


# ============================================================
# 10. RESUME
# ============================================================

def job_key(
    row,
):

    return (

        round(
            float(
                row[
                    "alpha_econ"
                ]
            ),
            2,
        ),

        round(
            float(
                row[
                    "alpha_soci"
                ]
            ),
            2,
        ),

        int(
            row[
                "question_idx"
            ]
        ),

        int(
            row[
                "rollout"
            ]
        ),
    )


existing_rows = []

completed = set()


if (
    RESUME
    and RAW_CSV.exists()
):

    old = pd.read_csv(
        RAW_CSV
    )


    required = {

        "alpha_econ",
        "alpha_soci",
        "question_idx",
        "rollout",
        "mean_entropy",
    }


    if required.issubset(
        old.columns
    ):


        old = (

            old
            .drop_duplicates(

                [
                    "alpha_econ",
                    "alpha_soci",
                    "question_idx",
                    "rollout",
                ],

                keep="last",
            )
        )


        existing_rows = (
            old.to_dict(
                "records"
            )
        )


        completed = {

            job_key(
                row
            )

            for row
            in existing_rows
        }


        print(

            "\nResume:",

            f"{len(completed):,}/"
            f"{EXPECTED_N:,}"
        )


PENDING_JOBS = [

    job

    for job
    in ALL_JOBS

    if (
        job_key(
            job
        )
        not in completed
    )
]


# ============================================================
# 11. LOAD MODEL
# ============================================================

if len(
    PENDING_JOBS
) > 0:


    from transformers import (

        AutoTokenizer,

        AutoModelForCausalLM,

        LogitsProcessor,

        LogitsProcessorList,
    )


    tokenizer = (

        AutoTokenizer
        .from_pretrained(

            MODEL_ID,

            trust_remote_code=
                TRUST_REMOTE_CODE,
        )
    )


    if (
        tokenizer.pad_token
        is None
    ):

        tokenizer.pad_token = (
            tokenizer.eos_token
        )

        tokenizer.pad_token_id = (
            tokenizer.eos_token_id
        )


    tokenizer.padding_side = (
        "left"
    )


    model = (

        AutoModelForCausalLM
        .from_pretrained(

            MODEL_ID,

            torch_dtype=
                torch.bfloat16,

            device_map={
                "":
                    "cuda:0"
            },

            trust_remote_code=
                TRUST_REMOTE_CODE,

            attn_implementation=
                "sdpa",
        )
    )


    model.eval()


    model.config.use_cache = True


    # --------------------------------------------------------
    # Keep orthogonal vectors resident on GPU.
    # --------------------------------------------------------

    v_e_gpu = (

        v_e_orth

        .to(

            device=
                device,

            dtype=
                torch.bfloat16,
        )
    )


    v_s_gpu = (

        v_s_orth

        .to(

            device=
                device,

            dtype=
                torch.bfloat16,
        )
    )


    torch.cuda.reset_peak_memory_stats()


    cuda_report(
        "after model load"
    )


    # ========================================================
    # 12. AUTO BATCH SIZE
    # ========================================================

    free_gb = (

        torch.cuda
        .mem_get_info()[0]

        / 2**30
    )


    if free_gb >= 100:

        BATCH_SIZE = min(
            TARGET_BATCH_SIZE,
            128,
        )


    elif free_gb >= 55:

        BATCH_SIZE = min(
            TARGET_BATCH_SIZE,
            64,
        )


    elif free_gb >= 28:

        BATCH_SIZE = min(
            TARGET_BATCH_SIZE,
            32,
        )


    elif free_gb >= 15:

        BATCH_SIZE = min(
            TARGET_BATCH_SIZE,
            16,
        )


    else:

        BATCH_SIZE = (
            MIN_BATCH_SIZE
        )


    print(
        "\nInitial BATCH_SIZE =",
        BATCH_SIZE
    )


    # ========================================================
    # 13. PRETOKENIZE QUESTIONS ONCE
    # ========================================================

    PRETOKENIZED = []


    for question in (
        NEUTRAL_QUESTIONS
    ):


        messages = [

            {

                "role":
                    "user",

                "content":
                    question
                    + PROMPT_SUFFIX,
            }
        ]


        prompt_text = (

            tokenizer
            .apply_chat_template(

                messages,

                tokenize=
                    False,

                add_generation_prompt=
                    True,
            )
        )


        ids = tokenizer(

            prompt_text,

            add_special_tokens=
                False,

            return_attention_mask=
                False,

        )[
            "input_ids"
        ]


        PRETOKENIZED.append(

            torch.tensor(

                ids,

                dtype=
                    torch.long,
            )
        )


    def collate_questions(
        question_indices,
    ):


        seqs = [

            PRETOKENIZED[
                int(
                    idx
                )
            ]

            for idx
            in question_indices
        ]


        max_len = max(

            len(
                seq
            )

            for seq
            in seqs
        )


        batch_n = len(
            seqs
        )


        input_ids = torch.full(

            (
                batch_n,
                max_len,
            ),

            tokenizer.pad_token_id,

            dtype=
                torch.long,

            device=
                device,
        )


        attention_mask = torch.zeros(

            (
                batch_n,
                max_len,
            ),

            dtype=
                torch.long,

            device=
                device,
        )


        for i, ids_cpu in enumerate(
            seqs
        ):


            n = len(
                ids_cpu
            )


            input_ids[
                i,
                -n:
            ] = (

                ids_cpu
                .to(

                    device,

                    non_blocking=
                        True,
                )
            )


            attention_mask[
                i,
                -n:
            ] = 1


        return {

            "input_ids":
                input_ids,

            "attention_mask":
                attention_mask,
        }


    # ========================================================
    # 14. FAST ENTROPY RECORDER
    #
    # Instead of:
    #
    # output_scores=True
    #
    # and retaining all vocabulary logits,
    #
    # compute entropy immediately at each generation step.
    #
    # Stores only:
    #
    #     [batch]
    #
    # entropy vector per token position.
    # ========================================================

    class EntropyRecorder(
        LogitsProcessor
    ):


        def __init__(
            self,
            top_k,
        ):

            self.top_k = int(
                top_k
            )

            self.step_entropy = []


        def __call__(
            self,
            input_ids,
            scores,
        ):


            logits = (
                scores.float()
            )


            top_logits = (

                torch.topk(

                    logits,

                    k=
                        self.top_k,

                    dim=
                        -1,

                ).values
            )


            probabilities = (

                torch.softmax(

                    top_logits,

                    dim=
                        -1,
                )
            )


            entropy = -(

                probabilities

                * torch.log2(

                    probabilities
                    .clamp_min(
                        1e-12
                    )
                )

            ).sum(
                dim=-1
            )


            # No computation graph under inference_mode.
            self.step_entropy.append(

                entropy.detach()
            )


            return scores


    # ========================================================
    # 15. PER-ROW SYMMETRIC DUAL STEERER
    # ========================================================

    class PerRowOrthogonalSteerer:


        def __init__(

            self,

            model,

            alpha_e,

            alpha_s,

            layer_1based,
        ):


            self.model = model


            self.layer_idx = (

                int(
                    layer_1based
                )

                - 1
            )


            ae = torch.as_tensor(

                alpha_e,

                device=
                    device,

                dtype=
                    torch.bfloat16,
            )


            ac = torch.as_tensor(

                alpha_s,

                device=
                    device,

                dtype=
                    torch.bfloat16,
            )


            # -----------------------------------------------
            # Precompute the full intervention direction once
            # per batch.
            #
            # Because v_e and v_s are orthonormal:
            #
            # ||direction_i||
            # =
            # sqrt(ae_i^2 + as_i^2)
            #
            # independent of sign combination.
            # -----------------------------------------------

            self.direction = (

                ae[
                    :,
                    None
                ]

                * v_e_gpu[
                    None,
                    :
                ]

                +

                ac[
                    :,
                    None
                ]

                * v_s_gpu[
                    None,
                    :
                ]
            )


            self.handle = None


        def _hook(
            self,
            module,
            inputs,
            output,
        ):


            h = (

                output[0]

                if isinstance(
                    output,
                    tuple,
                )

                else output
            )


            batch_n = (
                h.shape[0]
            )


            if (
                batch_n
                != self.direction.shape[0]
            ):

                raise RuntimeError(

                    "Batch mismatch: "

                    f"hidden="
                    f"{batch_n}, "

                    f"direction="
                    f"{self.direction.shape[0]}"
                )


            h_new = (
                h.clone()
            )


            # Existing steering convention:
            #
            # perturbation magnitude scaled by current
            # residual-stream norm.
            original_norm = (

                h_new[
                    :,
                    -1,
                    :
                ]

                .norm(

                    dim=
                        -1,

                    keepdim=
                        True,
                )

                .clamp_min(
                    1e-8
                )
            )


            h_new[
                :,
                -1,
                :
            ] += (

                original_norm

                * self.direction
            )


            if isinstance(
                output,
                tuple,
            ):

                return (

                    (
                        h_new,
                    )

                    + output[
                        1:
                    ]
                )


            return h_new


        def __enter__(
            self,
        ):


            self.handle = (

                self.model
                .model
                .layers[
                    self.layer_idx
                ]

                .register_forward_hook(
                    self._hook
                )
            )


            return self


        def __exit__(
            self,
            exc_type,
            exc_value,
            traceback,
        ):


            if (
                self.handle
                is not None
            ):

                self.handle.remove()


            self.handle = None


            return False


    # ========================================================
    # 16. BATCH SEED
    # ========================================================

    def stable_batch_seed(
        jobs,
    ):


        text = "|".join(

            (
                f"{j['alpha_econ']:+.2f},"
                f"{j['alpha_soci']:+.2f},"
                f"{j['question_idx']},"
                f"{j['rollout']}"
            )

            for j
            in jobs
        )


        digest = (

            hashlib.sha256(
                text.encode()
            )

            .digest()
        )


        number = int.from_bytes(

            digest[
                :8
            ],

            "little",
        )


        return int(

            (
                BASE_SEED
                + number
            )

            % (
                2**63
                - 1
            )
        )


    # ========================================================
    # 17. EOS LENGTH
    # ========================================================

    def generated_lengths(
        generated,
    ):


        eos = (

            generated
            .eq(
                tokenizer.eos_token_id
            )
        )


        has_eos = (
            eos.any(
                dim=1
            )
        )


        first_eos = (

            eos.float()
            .argmax(
                dim=1
            )
            .long()

            + 1
        )


        maximum = torch.full_like(

            first_eos,

            generated.shape[1],
        )


        return torch.where(

            has_eos,

            first_eos,

            maximum,
        )


    # ========================================================
    # 18. RUN ONE GPU MICRO-BATCH
    # ========================================================

    @torch.inference_mode()
    def run_microbatch(
        jobs,
    ):


        q_indices = [

            job[
                "question_idx"
            ]

            for job
            in jobs
        ]


        alpha_e = [

            job[
                "alpha_econ"
            ]

            for job
            in jobs
        ]


        alpha_s = [

            job[
                "alpha_soci"
            ]

            for job
            in jobs
        ]


        inputs = (
            collate_questions(
                q_indices
            )
        )


        recorder = (
            EntropyRecorder(
                TOP_K_ENTROPY
            )
        )


        processors = (
            LogitsProcessorList(
                [
                    recorder
                ]
            )
        )


        seed = (
            stable_batch_seed(
                jobs
            )
        )


        try:


            with torch.random.fork_rng(

                devices=[
                    0
                ]
            ):


                torch.manual_seed(
                    seed
                )


                torch.cuda.manual_seed_all(
                    seed
                )


                with PerRowOrthogonalSteerer(

                    model,

                    alpha_e,

                    alpha_s,

                    COMMON_LAYER,

                ):


                    with torch.autocast(

                        device_type=
                            "cuda",

                        dtype=
                            torch.bfloat16,
                    ):


                        sequences = (

                            model.generate(

                                **inputs,

                                max_new_tokens=
                                    MAX_NEW_TOKENS,

                                do_sample=
                                    True,

                                temperature=
                                    1.0,

                                top_p=
                                    1.0,

                                use_cache=
                                    True,

                                logits_processor=
                                    processors,

                                return_dict_in_generate=
                                    False,

                                output_scores=
                                    False,

                                pad_token_id=
                                    tokenizer.pad_token_id,

                                eos_token_id=
                                    tokenizer.eos_token_id,
                            )
                        )


            input_width = (

                inputs[
                    "input_ids"
                ]

                .shape[1]
            )


            generated = (

                sequences[
                    :,
                    input_width:
                ]
            )


            lengths = (
                generated_lengths(
                    generated
                )
            )


            # -----------------------------------------------
            # [B, T]
            # -----------------------------------------------

            entropy_matrix = (

                torch.stack(

                    recorder.step_entropy,

                    dim=1,
                )
            )


            total_steps = (

                entropy_matrix
                .shape[1]
            )


            positions = (

                torch.arange(

                    total_steps,

                    device=
                        device,

                )[
                    None,
                    :
                ]
            )


            valid_mask = (

                positions

                < lengths[
                    :,
                    None
                ]
            )


            valid_float = (
                valid_mask.float()
            )


            # Full generated response.
            mean_entropy = (

                (
                    entropy_matrix
                    * valid_float
                )

                .sum(
                    dim=1
                )

                /

                valid_float
                .sum(
                    dim=1
                )
                .clamp_min(1)
            )


            # First 20 generated positions.
            mask20 = (

                valid_mask

                &

                (
                    positions
                    < 20
                )
            )


            m20 = mask20.float()


            entropy20 = (

                (
                    entropy_matrix
                    * m20
                )

                .sum(
                    dim=1
                )

                /

                m20.sum(
                    dim=1
                )
                .clamp_min(1)
            )


            # First 64.
            mask64 = (

                valid_mask

                &

                (
                    positions
                    < 64
                )
            )


            m64 = (
                mask64.float()
            )


            entropy64 = (

                (
                    entropy_matrix
                    * m64
                )

                .sum(
                    dim=1
                )

                /

                m64.sum(
                    dim=1
                )
                .clamp_min(1)
            )


            mean_cpu = (

                mean_entropy
                .detach()
                .cpu()
                .numpy()
            )


            e20_cpu = (

                entropy20
                .detach()
                .cpu()
                .numpy()
            )


            e64_cpu = (

                entropy64
                .detach()
                .cpu()
                .numpy()
            )


            lengths_cpu = (

                lengths
                .detach()
                .cpu()
                .numpy()
            )


            records = []


            for (
                job,
                h,
                h20,
                h64,
                length,
            ) in zip(

                jobs,
                mean_cpu,
                e20_cpu,
                e64_cpu,
                lengths_cpu,
            ):


                ae = float(
                    job[
                        "alpha_econ"
                    ]
                )


                ac = float(
                    job[
                        "alpha_soci"
                    ]
                )


                # Because corrected vectors are orthonormal.
                coefficient_norm = (
                    math.sqrt(
                        ae**2
                        + ac**2
                    )
                )


                records.append({

                    **job,

                    "mean_entropy":
                        float(h),

                    "entropy_first20":
                        float(h20),

                    "entropy_first64":
                        float(h64),

                    "n_tokens":
                        int(length),

                    "truncated":
                        bool(
                            int(length)
                            >= MAX_NEW_TOKENS
                        ),

                    "coefficient_norm":
                        coefficient_norm,

                    "entropy_definition":
                        ENTROPY_DEFINITION,

                    "seed":
                        seed,
                })


            # -----------------------------------------------
            # Delete GPU temporaries.
            # -----------------------------------------------

            del (
                sequences,
                generated,
                lengths,
                entropy_matrix,
                mean_entropy,
                entropy20,
                entropy64,
                valid_mask,
                valid_float,
                mask20,
                mask64,
                m20,
                m64,
                inputs,
                recorder,
                processors,
            )


            return records


        except Exception:


            try:
                del inputs
            except Exception:
                pass


            raise


    # ========================================================
    # 19. FAST GPU LOOP + OOM FALLBACK
    # ========================================================

    rows = (
        existing_rows.copy()
    )


    pending = (
        PENDING_JOBS.copy()
    )


    position = 0

    successful_batches = 0

    current_bs = (
        BATCH_SIZE
    )


    pbar = tqdm(

        total=
            len(
                pending
            ),

        desc=
            "SymOrth 2D grid",

        unit=
            "seq",
    )


    while (
        position
        < len(
            pending
        )
    ):


        take = min(

            current_bs,

            len(
                pending
            )
            - position,
        )


        jobs = (

            pending[
                position:
                position
                + take
            ]
        )


        try:


            new_records = (
                run_microbatch(
                    jobs
                )
            )


            rows.extend(
                new_records
            )


            position += take


            pbar.update(
                take
            )


            successful_batches += 1


            # -----------------------------------------------
            # Atomic checkpoint.
            # -----------------------------------------------

            if (

                successful_batches
                % CHECKPOINT_EVERY
                == 0
            ):


                temp = (

                    RAW_CSV
                    .with_suffix(
                        ".tmp.csv"
                    )
                )


                (

                    pd.DataFrame(
                        rows
                    )

                    .drop_duplicates(

                        [
                            "alpha_econ",
                            "alpha_soci",
                            "question_idx",
                            "rollout",
                        ],

                        keep=
                            "last",
                    )

                    .to_csv(

                        temp,

                        index=False,
                    )
                )


                os.replace(

                    temp,

                    RAW_CSV,
                )


            # -----------------------------------------------
            # Periodic CUDA housekeeping.
            # -----------------------------------------------

            if (

                successful_batches
                % CUDA_CLEAN_EVERY
                == 0
            ):


                gc.collect()

                torch.cuda.empty_cache()


                cuda_report(

                    f"batch "
                    f"{successful_batches}"
                )


        except torch.cuda.OutOfMemoryError:


            print(

                "\nCUDA OOM at batch =",
                take
            )


            gc.collect()

            torch.cuda.empty_cache()


            if (
                current_bs
                <= MIN_BATCH_SIZE
            ):

                raise


            current_bs = max(

                MIN_BATCH_SIZE,

                current_bs
                // 2,
            )


            print(

                "Retrying with BATCH_SIZE =",
                current_bs
            )


    pbar.close()


    # ========================================================
    # 20. FINAL RAW DATA
    # ========================================================

    df_raw = (

        pd.DataFrame(
            rows
        )

        .drop_duplicates(

            [
                "alpha_econ",
                "alpha_soci",
                "question_idx",
                "rollout",
            ],

            keep=
                "last",
        )

        .sort_values(

            [
                "alpha_econ",
                "alpha_soci",
                "question_idx",
                "rollout",
            ]
        )

        .reset_index(
            drop=True
        )
    )


    df_raw.to_csv(

        RAW_CSV,

        index=False,
    )


    assert (
        len(
            df_raw
        )
        == EXPECTED_N
    ), (

        f"Expected "
        f"{EXPECTED_N}, "

        f"got "
        f"{len(df_raw)}"
    )


    print(
        "\nGeneration complete."
    )


    print(
        "rows =",
        f"{len(df_raw):,}"
    )


    print(

        "truncation =",

        f"{df_raw['truncated'].mean():.2%}"
    )


    cuda_report(
        "generation complete"
    )


else:


    print(
        "\nAll generations already exist."
    )


    df_raw = (
        pd.read_csv(
            RAW_CSV
        )
    )


# ============================================================
# 21. QUESTION-LEVEL UNIT
#
# Average the 5 rollouts.
# ============================================================

df_raw[
    "alpha_econ"
] = (

    df_raw[
        "alpha_econ"
    ]

    .round(2)
)


df_raw[
    "alpha_soci"
] = (

    df_raw[
        "alpha_soci"
    ]

    .round(2)
)


unit = (

    df_raw

    .groupby(

        [
            "alpha_econ",
            "alpha_soci",
            "question_idx",
        ],

        as_index=False,
    )

    .agg(

        mean_entropy=(
            "mean_entropy",
            "mean",
        ),

        entropy_first20=(
            "entropy_first20",
            "mean",
        ),

        entropy_first64=(
            "entropy_first64",
            "mean",
        ),

        rollout_sd=(
            "mean_entropy",
            "std",
        ),

        mean_tokens=(
            "n_tokens",
            "mean",
        ),

        truncation_rate=(
            "truncated",
            "mean",
        ),

        n_rollouts=(
            "rollout",
            "count",
        ),
    )
)


unit.to_csv(

    UNIT_CSV,

    index=False,
)


print(

    "\nQuestion-level rows:",

    len(
        unit
    )
)


# ============================================================
# 22. HELPERS
# ============================================================

def safe_wilcoxon_greater(
    values,
):


    values = (

        pd.Series(
            values
        )

        .dropna()

        .to_numpy(
            float
        )
    )


    if (
        len(
            values
        )
        == 0
    ):

        return (
            np.nan,
            np.nan,
        )


    if np.allclose(
        values,
        0,
    ):

        return (
            0.0,
            1.0,
        )


    result = wilcoxon(

        values,

        alternative=
            "greater",

        zero_method=
            "wilcox",
    )


    return (

        float(
            result.statistic
        ),

        float(
            result.pvalue
        ),
    )


def cell_series(
    metric,
    alpha_e,
    alpha_s,
):


    return (

        unit[

            np.isclose(
                unit[
                    "alpha_econ"
                ],
                alpha_e,
            )

            &

            np.isclose(
                unit[
                    "alpha_soci"
                ],
                alpha_s,
            )
        ]

        .set_index(
            "question_idx"
        )[
            metric
        ]
    )


# ============================================================
# 23. SINGLE-AXIS POLARITY DIAGNOSTIC
#
# Important:
# After fixing geometry, check whether ± direction itself
# still creates asymmetric entropy.
# ============================================================

MAGNITUDES = [

    0.05,
    0.10,
    0.15,
    0.20,
]


polarity_rows = []


baseline = (
    cell_series(
        "mean_entropy",
        0,
        0,
    )
)


print(
    "\n"
    + "=" * 100
)


print(
    "SINGLE-AXIS POLARITY DIAGNOSTIC"
)


print(
    "=" * 100
)


for a in MAGNITUDES:


    e_left = (
        cell_series(
            "mean_entropy",
            +a,
            0,
        )
    )


    e_right = (
        cell_series(
            "mean_entropy",
            -a,
            0,
        )
    )


    s_prog = (
        cell_series(
            "mean_entropy",
            0,
            +a,
        )
    )


    s_trad = (
        cell_series(
            "mean_entropy",
            0,
            -a,
        )
    )


    idx = (

        baseline.index

        .intersection(
            e_left.index
        )

        .intersection(
            e_right.index
        )

        .intersection(
            s_prog.index
        )

        .intersection(
            s_trad.index
        )
    )


    rows_a = {

        "alpha_mag":
            a,

        "EL_minus_baseline":
            (
                e_left.loc[idx]
                - baseline.loc[idx]
            ).mean(),

        "ER_minus_baseline":
            (
                e_right.loc[idx]
                - baseline.loc[idx]
            ).mean(),

        "SP_minus_baseline":
            (
                s_prog.loc[idx]
                - baseline.loc[idx]
            ).mean(),

        "ST_minus_baseline":
            (
                s_trad.loc[idx]
                - baseline.loc[idx]
            ).mean(),
    }


    polarity_rows.append(
        rows_a
    )


    print(

        f"a={a:.2f} | "

        f"EL="
        f"{rows_a['EL_minus_baseline']:+.4f} | "

        f"ER="
        f"{rows_a['ER_minus_baseline']:+.4f} | "

        f"SP="
        f"{rows_a['SP_minus_baseline']:+.4f} | "

        f"ST="
        f"{rows_a['ST_minus_baseline']:+.4f}"
    )


polarity_df = pd.DataFrame(
    polarity_rows
)


polarity_df.to_csv(

    RUN_DIR
    / "single_axis_polarity_diagnostic.csv",

    index=False,
)


# ============================================================
# 24. PRIMARY STRONG SYMMETRIC SIMPLE EFFECTS
#
# H1a:
#
#   EL-ST - EL-SP > 0
#
# H1b:
#
#   ER-SP - ER-ST > 0
#
# ============================================================

strong_rows = []


print(
    "\n"
    + "=" * 100
)


print(
    "STRONG SYMMETRIC CROSS-PRESSURE"
)


print(
    "=" * 100
)


for metric in [

    "mean_entropy",

    "entropy_first20",

    "entropy_first64",
]:


    for a in MAGNITUDES:


        EL_SP = (
            cell_series(
                metric,
                +a,
                +a,
            )
        )


        EL_ST = (
            cell_series(
                metric,
                +a,
                -a,
            )
        )


        ER_SP = (
            cell_series(
                metric,
                -a,
                +a,
            )
        )


        ER_ST = (
            cell_series(
                metric,
                -a,
                -a,
            )
        )


        idx = (

            EL_SP.index

            .intersection(
                EL_ST.index
            )

            .intersection(
                ER_SP.index
            )

            .intersection(
                ER_ST.index
            )
        )


        d_EL = (

            EL_ST.loc[
                idx
            ]

            -

            EL_SP.loc[
                idx
            ]
        )


        d_ER = (

            ER_SP.loc[
                idx
            ]

            -

            ER_ST.loc[
                idx
            ]
        )


        _, p_EL = (
            safe_wilcoxon_greater(
                d_EL
            )
        )


        _, p_ER = (
            safe_wilcoxon_greater(
                d_ER
            )
        )


        symmetric = (

            (
                d_EL > 0
            )

            &

            (
                d_ER > 0
            )
        )


        strong_rows.extend([


            {

                "metric":
                    metric,

                "alpha_mag":
                    a,

                "side":
                    "EL_ST_minus_EL_SP",

                "mean_delta":
                    d_EL.mean(),

                "median_delta":
                    d_EL.median(),

                "positive_share":
                    (
                        d_EL > 0
                    ).mean(),

                "p_one_sided":
                    p_EL,

                "n_questions":
                    len(
                        d_EL
                    ),
            },


            {

                "metric":
                    metric,

                "alpha_mag":
                    a,

                "side":
                    "ER_SP_minus_ER_ST",

                "mean_delta":
                    d_ER.mean(),

                "median_delta":
                    d_ER.median(),

                "positive_share":
                    (
                        d_ER > 0
                    ).mean(),

                "p_one_sided":
                    p_ER,

                "n_questions":
                    len(
                        d_ER
                    ),
            },
        ])


        if (
            metric
            == "mean_entropy"
        ):


            print(

                f"a={a:.2f} | "

                f"EL-ST − EL-SP = "
                f"{d_EL.mean():+.4f} "

                f"p={p_EL:.4g} | "

                f"ER-SP − ER-ST = "
                f"{d_ER.mean():+.4f} "

                f"p={p_ER:.4g} | "

                f"both="
                f"{symmetric.mean():.1%}"
            )


strong_results = pd.DataFrame(
    strong_rows
)


strong_results.to_csv(

    RUN_DIR
    / "strong_symmetric_simple_effects.csv",

    index=False,
)


# ============================================================
# 25. FOUR FIXED-AXIS SWEEPS
#
# Derived from grid. NO extra generation.
# ============================================================

fixed_rows = []


for fixed_axis, fixed_value in [

    (
        "econ",
        +ANCHOR,
    ),

    (
        "econ",
        -ANCHOR,
    ),

    (
        "soci",
        +ANCHOR,
    ),

    (
        "soci",
        -ANCHOR,
    ),
]:


    if (
        fixed_axis
        == "econ"
    ):


        d = (

            unit[

                np.isclose(

                    unit[
                        "alpha_econ"
                    ],

                    fixed_value,
                )
            ]

            .copy()
        )


        sweep_col = (
            "alpha_soci"
        )


    else:


        d = (

            unit[

                np.isclose(

                    unit[
                        "alpha_soci"
                    ],

                    fixed_value,
                )
            ]

            .copy()
        )


        sweep_col = (
            "alpha_econ"
        )


    # --------------------------------------------------------
    # Conflict progress:
    #
    # positive fixed anchor:
    # opposite sign => negative sweep.
    #
    # negative fixed anchor:
    # opposite sign => positive sweep.
    # --------------------------------------------------------

    d[
        "conflict_progress"
    ] = (

        -np.sign(
            fixed_value
        )

        * d[
            sweep_col
        ]
    )


    rho, p = (
        spearmanr(

            d[
                "conflict_progress"
            ],

            d[
                "mean_entropy"
            ],
        )
    )


    fixed_rows.append({

        "fixed_axis":
            fixed_axis,

        "fixed_value":
            fixed_value,

        "sweep_axis":
            sweep_col,

        "spearman_conflict_entropy":
            rho,

        "p":
            p,

        "n":
            len(
                d
            ),
    })


fixed_results = pd.DataFrame(
    fixed_rows
)


fixed_results.to_csv(

    RUN_DIR
    / "fixed_axis_sweep_tests.csv",

    index=False,
)


print(
    "\nFIXED AXIS SWEEPS"
)


display(
    fixed_results.round(5)
)


# ============================================================
# 26. NON-ADDITIVE ENTROPY
#
# R(ae,as)
# =
# H(ae,as)
# - H(ae,0)
# - H(0,as)
# + H(0,0)
# ============================================================

base_table = (

    unit[

        np.isclose(
            unit[
                "alpha_econ"
            ],
            0,
        )

        &

        np.isclose(
            unit[
                "alpha_soci"
            ],
            0,
        )
    ]

    [
        [
            "question_idx",
            "mean_entropy",
        ]
    ]

    .rename(

        columns={
            "mean_entropy":
                "H00"
        }
    )
)


econ_table = (

    unit[

        np.isclose(
            unit[
                "alpha_soci"
            ],
            0,
        )
    ]

    [
        [
            "alpha_econ",
            "question_idx",
            "mean_entropy",
        ]
    ]

    .rename(

        columns={
            "mean_entropy":
                "He0"
        }
    )
)


social_table = (

    unit[

        np.isclose(
            unit[
                "alpha_econ"
            ],
            0,
        )
    ]

    [
        [
            "alpha_soci",
            "question_idx",
            "mean_entropy",
        ]
    ]

    .rename(

        columns={
            "mean_entropy":
                "H0s"
        }
    )
)


residual = (

    unit[
        [
            "alpha_econ",
            "alpha_soci",
            "question_idx",
            "mean_entropy",
        ]
    ]

    .merge(

        econ_table,

        on=[
            "alpha_econ",
            "question_idx",
        ],

        how=
            "left",
    )

    .merge(

        social_table,

        on=[
            "alpha_soci",
            "question_idx",
        ],

        how=
            "left",
    )

    .merge(

        base_table,

        on=
            "question_idx",

        how=
            "left",
    )
)


residual[
    "interaction_residual"
] = (

    residual[
        "mean_entropy"
    ]

    -

    residual[
        "He0"
    ]

    -

    residual[
        "H0s"
    ]

    +

    residual[
        "H00"
    ]
)


residual.to_csv(

    RESID_CSV,

    index=False,
)


# ============================================================
# 27. NON-ADDITIVE CROSS VS CONSISTENT
# ============================================================

nonadd_rows = []


print(
    "\n"
    + "=" * 100
)


print(
    "NON-ADDITIVE CROSS-PRESSURE"
)


print(
    "=" * 100
)


def residual_cell(
    ae,
    ac,
):


    return (

        residual[

            np.isclose(
                residual[
                    "alpha_econ"
                ],
                ae,
            )

            &

            np.isclose(
                residual[
                    "alpha_soci"
                ],
                ac,
            )
        ]

        .set_index(
            "question_idx"
        )[
            "interaction_residual"
        ]
    )


for a in MAGNITUDES:


    EL_SP = (
        residual_cell(
            +a,
            +a,
        )
    )


    EL_ST = (
        residual_cell(
            +a,
            -a,
        )
    )


    ER_SP = (
        residual_cell(
            -a,
            +a,
        )
    )


    ER_ST = (
        residual_cell(
            -a,
            -a,
        )
    )


    idx = (

        EL_SP.index

        .intersection(
            EL_ST.index
        )

        .intersection(
            ER_SP.index
        )

        .intersection(
            ER_ST.index
        )
    )


    R_cross = (

        EL_ST.loc[
            idx
        ]

        +

        ER_SP.loc[
            idx
        ]

    ) / 2


    R_consistent = (

        EL_SP.loc[
            idx
        ]

        +

        ER_ST.loc[
            idx
        ]

    ) / 2


    difference = (

        R_cross

        -

        R_consistent
    )


    _, p = (
        safe_wilcoxon_greater(
            difference
        )
    )


    nonadd_rows.append({

        "alpha_mag":
            a,

        "cross_residual_mean":
            R_cross.mean(),

        "consistent_residual_mean":
            R_consistent.mean(),

        "cross_minus_consistent":
            difference.mean(),

        "positive_share":
            (
                difference > 0
            ).mean(),

        "p_one_sided":
            p,

        "n_questions":
            len(
                difference
            ),
    })


    print(

        f"a={a:.2f} | "

        f"R_cross="
        f"{R_cross.mean():+.4f} | "

        f"R_cons="
        f"{R_consistent.mean():+.4f} | "

        f"Δ="
        f"{difference.mean():+.4f} | "

        f"p="
        f"{p:.4g}"
    )


nonadd_results = pd.DataFrame(
    nonadd_rows
)


nonadd_results.to_csv(

    RUN_DIR
    / "nonadditive_diagonal_tests.csv",

    index=False,
)


# ============================================================
# 28. 2D RESPONSE-SURFACE REGRESSION
#
# Cross-pressure predicts:
#
#     beta_E×S < 0
# ============================================================

reg_df = (
    unit.copy()
)


fit = (

    smf.ols(

        (
            "mean_entropy ~ "
            "alpha_econ + "
            "alpha_soci + "
            "I(alpha_econ ** 2) + "
            "I(alpha_soci ** 2) + "
            "alpha_econ:alpha_soci + "
            "C(question_idx)"
        ),

        data=
            reg_df,
    )

    .fit(

        cov_type=
            "cluster",

        cov_kwds={

            "groups":
                reg_df[
                    "question_idx"
                ]
        },
    )
)


interaction_term = (
    "alpha_econ:alpha_soci"
)


interaction_table = pd.DataFrame([{

    "term":
        interaction_term,

    "coef":
        fit.params[
            interaction_term
        ],

    "se_cluster":
        fit.bse[
            interaction_term
        ],

    "p_two_sided":
        fit.pvalues[
            interaction_term
        ],

    "ci_low":
        fit.conf_int()
        .loc[
            interaction_term,
            0,
        ],

    "ci_high":
        fit.conf_int()
        .loc[
            interaction_term,
            1,
        ],
}])


interaction_table.to_csv(

    RUN_DIR
    / "surface_interaction_regression.csv",

    index=False,
)


print(
    "\n2D INTERACTION "
    "(hypothesis: coefficient < 0)"
)


display(
    interaction_table.round(6)
)


# ============================================================
# 29. HEATMAP — RAW ENTROPY
# ============================================================

surface = (

    unit

    .groupby(
        [
            "alpha_econ",
            "alpha_soci",
        ]
    )[
        "mean_entropy"
    ]

    .mean()

    .reset_index()
)


heat = (

    surface

    .pivot(

        index=
            "alpha_econ",

        columns=
            "alpha_soci",

        values=
            "mean_entropy",
    )

    .reindex(

        index=
            ALPHAS,

        columns=
            ALPHAS,
    )
)


fig, ax = plt.subplots(
    figsize=(
        7,
        6,
    )
)


image = ax.imshow(

    heat.values,

    origin=
        "lower",

    aspect=
        "auto",

    extent=[

        min(
            ALPHAS
        ),

        max(
            ALPHAS
        ),

        min(
            ALPHAS
        ),

        max(
            ALPHAS
        ),
    ],
)


ax.set_xlabel(
    "Social alpha (+ progressive, − traditional)"
)


ax.set_ylabel(
    "Economic alpha (+ left, − right)"
)


ax.set_title(
    "Symmetric-Orthogonalized 2D Steering: Mean Entropy"
)


fig.colorbar(

    image,

    ax=ax,

    label=
        "Mean top-40 entropy (bits)",
)


plt.tight_layout()


plt.savefig(

    RUN_DIR
    / "heatmap_raw_entropy_symorth.png",

    dpi=
        180,

    bbox_inches=
        "tight",
)


plt.show()


# ============================================================
# 30. HEATMAP — NON-ADDITIVE RESIDUAL
# ============================================================

resid_surface = (

    residual

    .groupby(
        [
            "alpha_econ",
            "alpha_soci",
        ]
    )[
        "interaction_residual"
    ]

    .mean()

    .reset_index()
)


resid_heat = (

    resid_surface

    .pivot(

        index=
            "alpha_econ",

        columns=
            "alpha_soci",

        values=
            "interaction_residual",
    )

    .reindex(

        index=
            ALPHAS,

        columns=
            ALPHAS,
    )
)


fig, ax = plt.subplots(
    figsize=(
        7,
        6,
    )
)


image = ax.imshow(

    resid_heat.values,

    origin=
        "lower",

    aspect=
        "auto",

    extent=[

        min(
            ALPHAS
        ),

        max(
            ALPHAS
        ),

        min(
            ALPHAS
        ),

        max(
            ALPHAS
        ),
    ],
)


ax.set_xlabel(
    "Social alpha (+ progressive, − traditional)"
)


ax.set_ylabel(
    "Economic alpha (+ left, − right)"
)


ax.set_title(
    "SymOrth Non-Additive Entropy Residual"
)


fig.colorbar(

    image,

    ax=ax,

    label=
        "Excess entropy beyond additive single-axis effects",
)


plt.tight_layout()


plt.savefig(

    RUN_DIR
    / "heatmap_nonadditive_residual_symorth.png",

    dpi=
        180,

    bbox_inches=
        "tight",
)


plt.show()


# ============================================================
# 31. FOUR FIXED-AXIS SWEEP PLOTS
# ============================================================

fig, axes = plt.subplots(

    2,
    2,

    figsize=(
        12,
        9,
    ),

    sharey=
        True,
)


plot_specs = [

    (
        "econ",
        +ANCHOR,
        axes[
            0,
            0
        ],
        "Econ +0.15 fixed; Social sweep",
    ),

    (
        "econ",
        -ANCHOR,
        axes[
            0,
            1
        ],
        "Econ −0.15 fixed; Social sweep",
    ),

    (
        "soci",
        +ANCHOR,
        axes[
            1,
            0
        ],
        "Social +0.15 fixed; Econ sweep",
    ),

    (
        "soci",
        -ANCHOR,
        axes[
            1,
            1
        ],
        "Social −0.15 fixed; Econ sweep",
    ),
]


for (
    fixed_axis,
    fixed_value,
    ax,
    title,
) in plot_specs:


    if (
        fixed_axis
        == "econ"
    ):


        d = unit[

            np.isclose(

                unit[
                    "alpha_econ"
                ],

                fixed_value,
            )
        ]


        x_col = (
            "alpha_soci"
        )


    else:


        d = unit[

            np.isclose(

                unit[
                    "alpha_soci"
                ],

                fixed_value,
            )
        ]


        x_col = (
            "alpha_econ"
        )


    summary = (

        d

        .groupby(
            x_col
        )[
            "mean_entropy"
        ]

        .agg(
            [
                "mean",
                "sem",
            ]
        )

        .reindex(
            ALPHAS
        )
    )


    ax.errorbar(

        summary.index,

        summary[
            "mean"
        ],

        yerr=
            1.96

            * summary[
                "sem"
            ],

        marker=
            "o",

        capsize=
            3,
    )


    ax.axvline(

        0,

        linewidth=
            1,

        alpha=
            0.4,
    )


    ax.set_title(
        title
    )


    ax.set_xlabel(
        x_col
    )


    ax.set_ylabel(
        "Mean entropy"
    )


    ax.grid(
        alpha=
            0.2
    )


plt.tight_layout()


plt.savefig(

    RUN_DIR
    / "four_symmetric_fixed_axis_sweeps_symorth.png",

    dpi=
        180,

    bbox_inches=
        "tight",
)


plt.show()


# ============================================================
# 32. STRONG SIMPLE-EFFECT PLOT
# ============================================================

plot_strong = (

    strong_results[

        strong_results[
            "metric"
        ]
        == "mean_entropy"
    ]
)


fig, ax = plt.subplots(
    figsize=(
        7.5,
        5,
    )
)


for side, label in [

    (
        "EL_ST_minus_EL_SP",
        "EL-ST minus EL-SP",
    ),

    (
        "ER_SP_minus_ER_ST",
        "ER-SP minus ER-ST",
    ),
]:


    d = (

        plot_strong[

            plot_strong[
                "side"
            ]
            == side
        ]

        .sort_values(
            "alpha_mag"
        )
    )


    ax.plot(

        d[
            "alpha_mag"
        ],

        d[
            "mean_delta"
        ],

        marker=
            "o",

        label=
            label,
    )


ax.axhline(

    0,

    linewidth=
        1,
)


ax.set_xlabel(
    "|alpha| on both orthogonalized axes"
)


ax.set_ylabel(
    "Cross − corresponding consistent entropy"
)


ax.set_title(
    "SymOrth Strong Cross-Pressure Simple Effects"
)


ax.legend()


ax.grid(
    alpha=
        0.2
)


plt.tight_layout()


plt.savefig(

    RUN_DIR
    / "strong_simple_effects_symorth.png",

    dpi=
        180,

    bbox_inches=
        "tight",
)


plt.show()


# ============================================================
# 33. FINAL DASHBOARD
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "CORRECTED MAIN ANALYSIS DASHBOARD"
)


print(
    "=" * 100
)


print(
    "\nVECTOR GEOMETRY"
)


print(

    "Raw cosine:",

    f"{RAW_COSINE:+.6f}"
)


print(

    "Corrected cosine:",

    f"{ORTH_COSINE:+.8f}"
)


print(
    "\nCORNER NORMS"
)


display(
    geometry_df.round(6)
)


print(
    "\nSINGLE-AXIS POLARITY"
)


display(
    polarity_df.round(5)
)


print(
    "\nSTRONG SIMPLE EFFECTS — FULL RESPONSE ENTROPY"
)


display(

    strong_results[

        strong_results[
            "metric"
        ]
        == "mean_entropy"
    ]

    .round(5)
)


print(
    "\nFIXED-AXIS SWEEPS"
)


display(
    fixed_results.round(5)
)


print(
    "\nNON-ADDITIVE CROSS VS CONSISTENT"
)


display(
    nonadd_results.round(5)
)


print(
    "\n2D INTERACTION"
)


display(
    interaction_table.round(6)
)


print(
    "\n"
    + "-" * 80
)


print(
    "HYPOTHESIS TARGETS"
)


print(
    "-" * 80
)


print(
    "1. EL-ST − EL-SP > 0"
)


print(
    "2. ER-SP − ER-ST > 0"
)


print(
    "3. BOTH simple effects > 0"
)


print(
    "4. R_cross − R_consistent > 0"
)


print(
    "5. alpha_econ × alpha_social coefficient < 0"
)


print(
    "\nOutputs:"
)


print(
    RUN_DIR
)


# ============================================================
# 34. CUDA CLEANUP
# ============================================================

if (
    len(
        PENDING_JOBS
    )
    > 0

    and

    FREE_MODEL_AT_END
):


    try:


        del (
            model,
            tokenizer,
            v_e_gpu,
            v_s_gpu,
        )


    except Exception:

        pass


    gc.collect()

    torch.cuda.empty_cache()


    try:

        torch.cuda.ipc_collect()

    except Exception:

        pass


    cuda_report(
        "after cleanup"
    )


print(
    "\nDONE."
)